In [45]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

## loading the trained model, scaler,pickle, onehot

In [46]:
model = load_model('model.h5')

## load the encder and scaler
with open('onehot_encoder_geography.pkl', 'rb') as f:
    onehot_encoder_geography = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

In [47]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [48]:
## one hot encode the geography column

geography_encoded = onehot_encoder_geography.transform(
    pd.DataFrame([[input_data['Geography']]], columns=['Geography'])
)

geo_encoded_df = pd.DataFrame(
    geography_encoded,
    columns=onehot_encoder_geography.get_feature_names_out(['Geography'])
)

In [49]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [50]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


## concatinating geography in input data

In [51]:
input_df=pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


## encoding categorical values

In [52]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [53]:
## scale the input data
scaled_input_data = scaler.transform(input_df)

In [ ]:

scaled_input_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [56]:
## predict churn probability
prediction=model.predict(scaled_input_data)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


array([[0.02674545]], dtype=float32)

In [57]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.026745452)

In [58]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
